#  Face Feature Extraction

## Objective

The objective of this notebook is to extract discriminative facial features from the cropped face images using two traditional computer vision techniques: Histogram of Oriented Gradients (HOG) and Local Binary Patterns (LBP).

These techniques will convert facial images into numerical feature vectors that can be used in subsequent face verification and machine learning stages.

## Feature Extraction Techniques

- **HOG (Histogram of Oriented Gradients):** Captures facial shape information through local gradient orientations and edge structures.
- **LBP (Local Binary Patterns):** Captures local texture information by analyzing relationships between neighboring pixel intensities.

## Input Dataset

The cropped and detected face images generated in the previous notebook:

`Dataset/Processed/Face/Face_Cropped/`

## Expected Outputs

- HOG feature vectors.
- LBP feature vectors.
- Extracted feature datasets with corresponding identity labels.

## 1. Environment & Configuration


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from skimage.feature import hog, local_binary_pattern
from skimage.color import rgb2gray

# Project Configuration
# The original project path was hard-coded to one computer. Run this notebook from the project root instead.
PROJECT_PATH = Path.cwd().resolve()

# Input Dataset
INPUT_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Cropped"
)

# Output Dataset
OUTPUT_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Features"
)

# User Configuration
USERS = ["User_01", "User_02"]

print("Configuration completed successfully.")
print("Input Dataset Exists:", INPUT_PATH.exists())
print("Output Dataset Path:", OUTPUT_PATH)

## 2. Dataset Loading & Validation


In [ ]:
# Load cropped face images

image_paths = {}
dataset_records = []

for user in USERS:

    user_path = INPUT_PATH / user

    images = sorted([
        path for path in user_path.iterdir()
        if path.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ])

    image_paths[user] = images

    print(f"{user}: {len(images)} images found.")

    for image_path in images:

        try:
            with Image.open(image_path) as image:
                image.verify()

            with Image.open(image_path) as image:
                image_width, image_height = image.size

            dataset_records.append({
                "image_name": image_path.name,
                "identity": user,
                "image_path": image_path,
                "width": image_width,
                "height": image_height
            })

        except Exception as error:
            print(f"WARNING: {image_path.name} - {error}")

# Create dataset DataFrame
face_dataset = pd.DataFrame(dataset_records)

print("\nDataset Validation Summary")
print("-" * 40)

print("Total Valid Images:", len(face_dataset))
print("\nImages per Identity:")
print(face_dataset["identity"].value_counts())

print("\nImage Dimensions:")
print(
    face_dataset[["width", "height"]]
    .describe()
    .round(2)
)

## 3. Grayscale Conversion

In [ ]:
# Configure Grayscale Standardization Output Directory

GRAYSCALE_STANDARDIZATION_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Grayscale_Standardization"
)

for user in USERS:
    (GRAYSCALE_STANDARDIZATION_PATH / user).mkdir(
        parents=True,
        exist_ok=True
    )

print("Grayscale Standardization output directories created successfully.")
print("Output Path:", GRAYSCALE_STANDARDIZATION_PATH)

In [ ]:
# Convert all cropped face images to Grayscale

grayscale_results = {}

for user in USERS:

    grayscale_results[user] = []

    for image_path in image_paths[user]:

        # Load RGB face image
        image = Image.open(image_path).convert("RGB")

        # Convert to Grayscale
        grayscale_image = image.convert("L")

        # Generate output path
        output_path = (
            GRAYSCALE_PATH
            / user
            / image_path.name
        )

        # Save Grayscale image
        grayscale_image.save(output_path)

        grayscale_results[user].append(output_path)

    print(
        f"{user}: {len(grayscale_results[user])} "
        "Grayscale images saved."
    )

print("\nGrayscale conversion completed successfully.")

## 4. HOG Feature Extraction

Histogram of Oriented Gradients (HOG) is a traditional feature extraction technique used to describe the shape and structural information of an image.

### 4.1 HOG Configuration

In this section, we define the HOG parameters used to extract facial gradient features..

In [ ]:
# HOG Configuration

HOG_ORIENTATIONS = 9

HOG_PIXELS_PER_CELL = (8, 8)

HOG_CELLS_PER_BLOCK = (2, 2)

HOG_BLOCK_NORM = "L2-Hys"

print("HOG Configuration")
print("-" * 35)
print("Orientations:", HOG_ORIENTATIONS)
print("Pixels per Cell:", HOG_PIXELS_PER_CELL)
print("Cells per Block:", HOG_CELLS_PER_BLOCK)
print("Block Normalization:", HOG_BLOCK_NORM)

### 4.2 Extract HOG Features

In this section, we apply the configured HOG descriptor to all grayscale face images.

In [ ]:
# HOG Feature Extraction

hog_features = []
hog_labels = []
hog_image_names = []

for user in USERS:

    user_path = GRAYSCALE_STANDARDIZATION_PATH / user

    image_paths = sorted([
        path for path in user_path.iterdir()
        if path.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ])

    for image_path in image_paths:

        image = Image.open(image_path).convert("L")
        image_array = np.array(image)

        features = hog(
            image_array,
            orientations=HOG_ORIENTATIONS,
            pixels_per_cell=HOG_PIXELS_PER_CELL,
            cells_per_block=HOG_CELLS_PER_BLOCK,
            block_norm=HOG_BLOCK_NORM,
            feature_vector=True
        )

        hog_features.append(features)
        hog_labels.append(user)
        hog_image_names.append(image_path.name)

# Verify feature vector lengths before conversion
feature_lengths = set(len(features) for features in hog_features)

print("Unique HOG Feature Vector Lengths:", feature_lengths)

if len(feature_lengths) != 1:
    raise ValueError("HOG feature vectors have inconsistent lengths.")

hog_features = np.stack(hog_features)

print("\nHOG Feature Extraction Completed")
print("-" * 40)
print("Total Images:", len(hog_features))
print("Feature Shape:", hog_features.shape)
print("Total Features per Image:", hog_features.shape[1])

## 5. LBP Feature Extraction

Local Binary Patterns (LBP) is a texture descriptor used to extract local
spatial patterns from grayscale images.

In [ ]:
# Configure LBP Parameters

LBP_POINTS = 8
LBP_RADIUS = 1
LBP_METHOD = "uniform"

print("LBP Configuration")
print("-" * 40)
print("Points:", LBP_POINTS)
print("Radius:", LBP_RADIUS)
print("Method:", LBP_METHOD)

In [ ]:
# Extract LBP Features

lbp_features = []
lbp_labels = []
lbp_image_names = []

for user in USERS:

    user_path = GRAYSCALE_STANDARDIZATION_PATH / user

    image_paths = sorted([
        path for path in user_path.iterdir()
        if path.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ])

    for image_path in image_paths:

        image = Image.open(image_path).convert("L")
        image_array = np.array(image)

        # Apply LBP operator
        lbp_image = local_binary_pattern(
            image_array,
            P=LBP_POINTS,
            R=LBP_RADIUS,
            method=LBP_METHOD
        )

        # Calculate normalized LBP histogram
        n_bins = LBP_POINTS + 2

        histogram, _ = np.histogram(
            lbp_image.ravel(),
            bins=np.arange(0, n_bins + 1),
            range=(0, n_bins)
        )

        histogram = histogram.astype("float32")
        histogram /= histogram.sum()

        lbp_features.append(histogram)
        lbp_labels.append(user)
        lbp_image_names.append(image_path.name)

# Convert features into NumPy array
lbp_features = np.stack(lbp_features)

print("LBP Feature Extraction Completed")
print("-" * 40)
print("Total Images:", len(lbp_features))
print("Feature Shape:", lbp_features.shape)
print("Features per Image:", lbp_features.shape[1])

## 6. Feature Dataset Preparation


### DataFrames

In [ ]:
# Create HOG Feature DataFrame

hog_feature_columns = [
    f"HOG_{i}" for i in range(hog_features.shape[1])
]

hog_df = pd.DataFrame(
    hog_features,
    columns=hog_feature_columns
)

hog_df.insert(0, "Image_Name", hog_image_names)
hog_df.insert(0, "User_ID", hog_labels)


# Create LBP Feature DataFrame

lbp_feature_columns = [
    f"LBP_{i}" for i in range(lbp_features.shape[1])
]

lbp_df = pd.DataFrame(
    lbp_features,
    columns=lbp_feature_columns
)

lbp_df.insert(0, "Image_Name", lbp_image_names)
lbp_df.insert(0, "User_ID", lbp_labels)


print("Feature DataFrames Created Successfully")
print("-" * 45)

print("HOG Dataset Shape:", hog_df.shape)
print("LBP Dataset Shape:", lbp_df.shape)

###  Validate Feature Datasets

In [ ]:
# Validate HOG and LBP Datasets

print("FEATURE DATASET VALIDATION")
print("=" * 45)

# 1. Check dataset dimensions
print("\nDataset Shapes:")
print("HOG:", hog_df.shape)
print("LBP:", lbp_df.shape)

# 2. Check identity distribution
print("\nIdentity Distribution:")
print(hog_df["User_ID"].value_counts().sort_index())

# 3. Check image matching
matching_images = (
    hog_df["Image_Name"].equals(lbp_df["Image_Name"])
    and hog_df["User_ID"].equals(lbp_df["User_ID"])
)

print("\nHOG and LBP Records Match:", matching_images)

# 4. Check missing values
print("\nMissing Values:")
print("HOG:", hog_df.isnull().sum().sum())
print("LBP:", lbp_df.isnull().sum().sum())

# 5. Check invalid numerical values
hog_values = hog_df[hog_feature_columns].to_numpy()
lbp_values = lbp_df[lbp_feature_columns].to_numpy()

print("\nInvalid Numerical Values:")
print("HOG:", not np.isfinite(hog_values).all())
print("LBP:", not np.isfinite(lbp_values).all())

# 6. Check LBP histogram normalization
lbp_sums = lbp_values.sum(axis=1)

print("\nLBP Histogram Normalization:")
print("All Histograms Sum to 1:",
      np.allclose(lbp_sums, 1.0))

## 7. Save Extracted Features

### 7.1 – Configure Output Directory

In [ ]:
# Configure Face Features Output Directory

OUTPUT_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Features"
)

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Feature output directory created successfully.")
print("Output Path:", OUTPUT_PATH)

### 7.2: Save Feature Datasets

In [ ]:
# Save HOG and LBP Feature Datasets

HOG_OUTPUT_FILE = OUTPUT_PATH / "HOG_Features.csv"
LBP_OUTPUT_FILE = OUTPUT_PATH / "LBP_Features.csv"

hog_df.to_csv(
    HOG_OUTPUT_FILE,
    index=False
)

lbp_df.to_csv(
    LBP_OUTPUT_FILE,
    index=False
)

print("Feature Datasets Saved Successfully")
print("-" * 45)
print("HOG File:", HOG_OUTPUT_FILE)
print("LBP File:", LBP_OUTPUT_FILE)

### Combine HOG and LBP Features

In [ ]:
# Combine HOG and LBP Feature Datasets

if not (
    hog_df["User_ID"].equals(lbp_df["User_ID"])
    and hog_df["Image_Name"].equals(lbp_df["Image_Name"])
):
    raise ValueError("HOG and LBP records do not match.")

face_combined_df = pd.concat(
    [
        hog_df[["User_ID", "Image_Name"]],
        hog_df[hog_feature_columns],
        lbp_df[lbp_feature_columns]
    ],
    axis=1
)

# Save Combined Face Features
COMBINED_OUTPUT_FILE = OUTPUT_PATH / "Face_Combined_Features.csv"

face_combined_df.to_csv(
    COMBINED_OUTPUT_FILE,
    index=False
)

print("Combined Face Dataset Created Successfully")
print("-" * 45)
print("Dataset Shape:", face_combined_df.shape)
print("Output File:", COMBINED_OUTPUT_FILE)

In [ ]:
# Combine HOG and LBP Feature Datasets

if not (
    hog_df["User_ID"].equals(lbp_df["User_ID"])
    and hog_df["Image_Name"].equals(lbp_df["Image_Name"])
):
    raise ValueError("HOG and LBP records do not match.")

face_combined_df = pd.concat(
    [
        hog_df[["User_ID", "Image_Name"]],
        hog_df[hog_feature_columns],
        lbp_df[lbp_feature_columns]
    ],
    axis=1
)

# Save Combined Face Features
COMBINED_OUTPUT_FILE = OUTPUT_PATH / "Face_Combined_Features.csv"

face_combined_df.to_csv(
    COMBINED_OUTPUT_FILE,
    index=False
)

print("Combined Face Dataset Created Successfully")
print("-" * 45)
print("Dataset Shape:", face_combined_df.shape)
print("Output File:", COMBINED_OUTPUT_FILE)

In [ ]:
# Combine Face Features: LBP First, Then HOG

face_combined_df = pd.concat(
    [
        hog_df[["User_ID", "Image_Name"]],
        lbp_df[lbp_feature_columns],
        hog_df[hog_feature_columns]
    ],
    axis=1
)

# Save Updated Combined Dataset

COMBINED_OUTPUT_FILE = OUTPUT_PATH / "Face_Combined_Features.csv"

face_combined_df.to_csv(
    COMBINED_OUTPUT_FILE,
    index=False
)

# Verify Column Order

print("First 15 Columns:")
print(face_combined_df.columns[:15].tolist())

print("\nDataset Shape:", face_combined_df.shape)
print("\nUpdated File:", COMBINED_OUTPUT_FILE)